# A/B Testing Statistics

## Learning Objectives
1. Implement the full z-test pipeline for comparing proportions including test statistic, p-value, and CI
2. Apply CUPED variance reduction using pre-experiment covariates and quantify the variance reduction
3. Simulate SPRT sequential testing to determine early stopping decisions
4. Implement Bayesian A/B testing with Beta-Binomial posteriors and compute P(B > A)

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
from typing import Tuple

np.random.seed(42)
print("numpy:", np.__version__)
print("scipy:", __import__('scipy').__version__)
print("Setup complete.")

## Level 1: Classic z-Test for Proportions

The two-sample z-test for proportions tests H0: p_A = p_B.

- Test statistic: z = (p_B - p_A) / SE_pooled
- SE_pooled = sqrt(p_hat * (1 - p_hat) * (1/n_A + 1/n_B))
- p_hat = (conversions_A + conversions_B) / (n_A + n_B) -- pooled proportion under H0

In [ ]:
# -----------------------------------------------------------------------
# Level 1: z-test for proportions + CI for the difference
# -----------------------------------------------------------------------

def two_proportion_z_test(
    conversions_a: int, n_a: int,
    conversions_b: int, n_b: int,
    alpha: float = 0.05
) -> dict:
    """
    Two-sample z-test for proportions (A/B test for conversion rates).

    Parameters
    ----------
    conversions_a, n_a : successes and total in group A (control)
    conversions_b, n_b : successes and total in group B (treatment)
    alpha              : significance level

    Returns
    -------
    dict with p_a, p_b, diff, z_stat, p_value, ci_lower, ci_upper, significant
    """
    p_a = conversions_a / n_a
    p_b = conversions_b / n_b
    diff = p_b - p_a

    # Pooled proportion under H0: p_A = p_B
    p_pooled = (conversions_a + conversions_b) / (n_a + n_b)
    se_pooled = np.sqrt(p_pooled * (1 - p_pooled) * (1 / n_a + 1 / n_b))

    # Test statistic
    z_stat = diff / se_pooled
    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))   # two-tailed

    # 95% CI for the difference (using unpooled SE for the CI)
    se_diff = np.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
    z_crit = stats.norm.ppf(1 - alpha / 2)
    ci_lower = diff - z_crit * se_diff
    ci_upper = diff + z_crit * se_diff

    return {
        "p_a": p_a, "p_b": p_b, "diff": diff,
        "se_pooled": se_pooled, "z_stat": z_stat, "p_value": p_value,
        "ci_lower": ci_lower, "ci_upper": ci_upper,
        "significant": p_value < alpha,
        "relative_lift": (p_b - p_a) / p_a * 100
    }


# Effect size (Cohen's h) for proportions
def cohens_h(p1: float, p2: float) -> float:
    """Cohen's h effect size for two proportions."""
    return abs(2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2)))


# ------ Scenario 1: Statistically significant improvement ------
result = two_proportion_z_test(480, 10000, 540, 10000)
print("Scenario 1: Baseline 4.8% -> Treatment 5.4%")
print(f"  p_A = {result['p_a']:.4f}, p_B = {result['p_b']:.4f}")
print(f"  Absolute diff: {result['diff']:+.4f} ({result['relative_lift']:+.1f}% relative lift)")
print(f"  z-statistic: {result['z_stat']:.3f}")
print(f"  p-value: {result['p_value']:.4f}  [significant: {result['significant']}]")
print(f"  95% CI for diff: [{result['ci_lower']:+.4f}, {result['ci_upper']:+.4f}]")
print(f"  CI excludes 0: {result['ci_lower'] > 0}")
print(f"  Cohen's h: {cohens_h(result['p_a'], result['p_b']):.4f}")

# ------ Scenario 2: No real effect ------
result2 = two_proportion_z_test(490, 10000, 510, 10000)
print(f"
Scenario 2: Baseline 4.9% -> Treatment 5.1% (tiny diff, n=10K)")
print(f"  p-value: {result2['p_value']:.4f}  [significant: {result2['significant']}]")
print(f"  95% CI: [{result2['ci_lower']:+.4f}, {result2['ci_upper']:+.4f}]  includes 0: {result2['ci_lower'] < 0}")

# ------ Scenario 3: Effect visible only with large n ------
result3 = two_proportion_z_test(4900, 100000, 5100, 100000)
print(f"
Scenario 3: Same 0.2pp diff, but n=100K per group")
print(f"  p-value: {result3['p_value']:.6f}  [significant: {result3['significant']}]")
print("  Large n detects tiny effects -- always check practical significance too")

## Level 2: Full A/B Test Analysis Pipeline

A production A/B test pipeline includes:
1. Effect size (Cohen's h) to quantify practical significance
2. Power analysis to verify the test was adequately powered
3. p-value and CI for the difference
4. Relative and absolute lift with uncertainty

In [ ]:
# -----------------------------------------------------------------------
# Level 2: Full A/B analysis pipeline with effect size, power, and reporting
# -----------------------------------------------------------------------

def full_ab_analysis(
    conversions_a: int, n_a: int,
    conversions_b: int, n_b: int,
    alpha: float = 0.05,
    target_power: float = 0.80,
    mde: float = None
) -> dict:
    """
    Full A/B test analysis with significance, effect size, and power check.

    Parameters
    ----------
    mde : minimum detectable effect (absolute pp) -- used for power check

    Returns
    -------
    dict with complete analysis results
    """
    # Core z-test
    z_result = two_proportion_z_test(conversions_a, n_a, conversions_b, n_b, alpha)

    # Effect size
    h = cohens_h(z_result['p_a'], z_result['p_b'])
    h_label = "trivial" if h < 0.1 else ("small" if h < 0.3 else
               ("medium" if h < 0.5 else "large"))

    # Post-hoc power (based on observed effect size)
    # n_harmonic for unequal groups
    n_harm = 2 / (1 / n_a + 1 / n_b)
    # ncp for the observed effect
    ncp_obs = h * np.sqrt(n_harm / 2)
    z_crit = stats.norm.ppf(1 - alpha / 2)
    power_obs = 1 - stats.norm.cdf(z_crit - ncp_obs)

    # MDE for the observed n
    if mde is None:
        # Back-compute MDE at 80% power
        z_alpha = stats.norm.ppf(1 - alpha / 2)
        z_beta = stats.norm.ppf(target_power)
        p_bar = (conversions_a + conversions_b) / (n_a + n_b)
        mde_computed = (z_alpha + z_beta) * np.sqrt(2 * p_bar * (1 - p_bar) / (n_harm / 2))
    else:
        mde_computed = mde

    z_result.update({
        "cohens_h": h,
        "h_label": h_label,
        "power_observed": power_obs,
        "mde": mde_computed,
        "n_a": n_a, "n_b": n_b,
        "practically_significant": abs(z_result['diff']) > mde_computed,
    })
    return z_result


def print_ab_report(result: dict) -> None:
    """Pretty-print a full A/B test report."""
    print(f"A/B Test Report")
    print(f"  Sample sizes:      A={result['n_a']:,}  B={result['n_b']:,}")
    print(f"  Conversion rates:  A={result['p_a']:.3%}  B={result['p_b']:.3%}")
    print(f"  Absolute lift:     {result['diff']:+.4%}  (relative: {result['relative_lift']:+.1f}%)")
    print(f"  95% CI for lift:   [{result['ci_lower']:+.4%}, {result['ci_upper']:+.4%}]")
    print(f"  z-statistic:       {result['z_stat']:.3f}")
    print(f"  p-value:           {result['p_value']:.4f}")
    print(f"  Statistically significant:  {result['significant']}")
    print(f"  Cohen's h:         {result['cohens_h']:.4f}  ({result['h_label']})")
    print(f"  Observed power:    {result['power_observed']:.1%}")
    print(f"  MDE (80% power):   {result['mde']:+.4%}")
    print(f"  Practically significant (|diff| > MDE): {result['practically_significant']}")


# Realistic A/B test: marketing funnel experiment
report = full_ab_analysis(
    conversions_a=950, n_a=20000,
    conversions_b=1100, n_b=20000,
    alpha=0.05, mde=0.005
)
print_ab_report(report)

print()
# Low-n scenario -- often mistakenly called "no effect"
report2 = full_ab_analysis(
    conversions_a=9, n_a=200,
    conversions_b=14, n_b=200,
    alpha=0.05
)
print("Small sample scenario (n=200 per group):")
print_ab_report(report2)
print("NOTE: Not significant but power is low -- cannot conclude 'no effect'")

## Real-World Example 1: CUPED Variance Reduction

CUPED (Controlled-experiment Using Pre-Experiment Data) reduces estimator variance by regressing out a pre-experiment covariate. The adjustment is:

Y_adjusted = Y - theta * (X - mean(X))

where theta = Cov(Y, X) / Var(X) and X is the pre-experiment metric.

Variance reduction = 1 - r^2(Y, X), where r is the correlation coefficient.

In [ ]:
# -----------------------------------------------------------------------
# Real-World Example 1: CUPED variance reduction
# -----------------------------------------------------------------------

def cuped_adjustment(y_treatment, y_control, x_treatment, x_control):
    """CUPED: Controlled-experiment Using Pre-Experiment Data.

    Adjusts outcomes by removing the component explained by pre-experiment data X.
    Reduces variance without introducing bias (same expected value as unadjusted).

    Parameters
    ----------
    y_treatment, y_control : post-experiment outcomes
    x_treatment, x_control : pre-experiment covariate (e.g. last week revenue)

    Returns
    -------
    (y_t_adj, y_c_adj, theta)
    """
    x_pooled = np.concatenate([x_treatment, x_control])
    y_pooled = np.concatenate([y_treatment, y_control])
    theta = np.cov(y_pooled, x_pooled)[0, 1] / np.var(x_pooled)
    y_t_adj = y_treatment - theta * (x_treatment - x_pooled.mean())
    y_c_adj = y_control - theta * (x_control - x_pooled.mean())
    return y_t_adj, y_c_adj, theta


# Simulate pre/post experiment data
rng = np.random.default_rng(42)
n = 2000
true_effect = 0.5   # true treatment lift
sigma = 5.0

# Pre-experiment metric (e.g. last week's revenue per user)
x_control = rng.normal(10, sigma, n)
x_treatment = rng.normal(10, sigma, n)   # no treatment effect in pre-period

# Post-experiment: correlated with pre-experiment (rho ~ 0.7)
rho = 0.70
noise_control   = rng.normal(0, sigma * np.sqrt(1 - rho**2), n)
noise_treatment = rng.normal(0, sigma * np.sqrt(1 - rho**2), n)
y_control   = rho * x_control   + noise_control
y_treatment = rho * x_treatment + noise_treatment + true_effect

# --- Standard t-test (unadjusted) ---
t_std, p_std = stats.ttest_ind(y_treatment, y_control)
diff_std = np.mean(y_treatment) - np.mean(y_control)
se_std = np.sqrt(np.var(y_treatment, ddof=1) / n + np.var(y_control, ddof=1) / n)

# --- CUPED adjusted ---
y_t_adj, y_c_adj, theta = cuped_adjustment(y_treatment, y_control, x_treatment, x_control)
t_cup, p_cup = stats.ttest_ind(y_t_adj, y_c_adj)
diff_cup = np.mean(y_t_adj) - np.mean(y_c_adj)
se_cup = np.sqrt(np.var(y_t_adj, ddof=1) / n + np.var(y_c_adj, ddof=1) / n)

# Theoretical variance reduction
empirical_corr = np.corrcoef(
    np.concatenate([y_treatment, y_control]),
    np.concatenate([x_treatment, x_control])
)[0, 1]
theoretical_reduction = 1 - empirical_corr ** 2

print(f"CUPED Analysis (n={n} per group, true effect={true_effect}, rho={rho})")
print(f"  Pre-post correlation (rho): {empirical_corr:.3f}")
print(f"  Theta (regression coefficient): {theta:.4f}")
print()
print(f"{'Metric':<30}  {'Standard':>12}  {'CUPED':>12}  {'Improvement':>14}")
print("-" * 72)
print(f"{'Estimated diff':<30}  {diff_std:>12.4f}  {diff_cup:>12.4f}  {'(same, unbiased)':>14}")
print(f"{'Standard error':<30}  {se_std:>12.4f}  {se_cup:>12.4f}  {(1 - se_cup/se_std)*100:>12.1f}% lower")
print(f"{'p-value':<30}  {p_std:>12.4f}  {p_cup:>12.4f}  {'(lower = more sensitive)':>14}")
print(f"{'t-statistic':<30}  {t_std:>12.3f}  {t_cup:>12.3f}  {t_cup/t_std - 1:>13.1%} larger")
print(f"{'Theoretical var reduction':<30}  {'N/A':>12}  {theoretical_reduction:>11.1%}  {'= 1 - rho^2':>14}")
print()
print(f"CUPED equivalent sample size gain: {1/(1 - empirical_corr**2):.2f}x more power")
print("(Same as running experiment with ~{:.0f}% more users)".format(
    (1/(1 - empirical_corr**2) - 1) * 100))

## Real-World Example 2: SPRT Sequential Testing

The Sequential Probability Ratio Test (SPRT) allows early stopping with valid Type I error control.
Unlike peeking at a fixed-horizon z-test, SPRT maintains the stated false positive rate across all looks.

Log-likelihood ratio: LLR = sum_i [obs_i * log(p1/p0) + (1-obs_i) * log((1-p1)/(1-p0))]
Stop when LLR >= A (reject H0) or LLR <= B (accept H0).

In [ ]:
# -----------------------------------------------------------------------
# Real-World Example 2: SPRT sequential testing
# -----------------------------------------------------------------------

def sprt_test(observations, p0, p1, alpha=0.05, beta=0.2):
    """SPRT: stop when LLR crosses bounds.

    Parameters
    ----------
    observations : sequence of binary outcomes (0/1)
    p0           : null hypothesis conversion rate
    p1           : alternative hypothesis conversion rate
    alpha        : Type I error rate (controls upper boundary A)
    beta         : Type II error rate = 1 - power (controls lower boundary B)

    Returns
    -------
    (decision, final_llr)
    """
    A = np.log((1 - beta) / alpha)    # upper bound -> reject H0
    B = np.log(beta / (1 - alpha))    # lower bound -> accept H0
    llr = 0
    for obs in observations:
        llr += obs * np.log(p1 / p0) + (1 - obs) * np.log((1 - p1) / (1 - p0))
        if llr >= A:
            return "reject H0", llr
        if llr <= B:
            return "accept H0", llr
    return "continue", llr


def simulate_sprt_path(n_obs: int, p_true: float, p0: float, p1: float,
                       alpha: float = 0.05, beta: float = 0.2) -> dict:
    """
    Simulate one SPRT run and return the full LLR path with stopping point.
    """
    rng = np.random.default_rng(None)  # different seed each call
    observations = rng.binomial(1, p_true, n_obs)
    A = np.log((1 - beta) / alpha)
    B = np.log(beta / (1 - alpha))

    llrs = [0]
    stop_at = None
    decision_taken = None

    llr = 0
    for i, obs in enumerate(observations):
        llr += obs * np.log(p1 / p0) + (1 - obs) * np.log((1 - p1) / (1 - p0))
        llrs.append(llr)
        if llr >= A and stop_at is None:
            stop_at = i + 1
            decision_taken = "reject H0"
            break
        if llr <= B and stop_at is None:
            stop_at = i + 1
            decision_taken = "accept H0"
            break

    return {"llrs": llrs, "stop_at": stop_at, "decision": decision_taken, "A": A, "B": B}


# Test SPRT function
p0, p1 = 0.05, 0.08   # H0: 5% conversion, H1: 8% conversion
alpha, beta = 0.05, 0.20
A = np.log((1 - beta) / alpha)
B = np.log(beta / (1 - alpha))
print(f"SPRT bounds: A={A:.3f} (reject H0), B={B:.3f} (accept H0)")

# Simulate 5 SPRT paths to visualize variation
rng_main = np.random.default_rng(42)
n_max = 3000
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax1 = axes[0]

stop_points_h1 = []
for trial in range(30):
    # H1 true (p_true = p1 = 0.08)
    obs = rng_main.binomial(1, p1, n_max)
    path = simulate_sprt_path(n_max, p1, p0, p1)
    if trial < 5:  # plot first 5
        ax1.plot(path["llrs"], alpha=0.6, linewidth=1)
    if path["stop_at"]:
        stop_points_h1.append(path["stop_at"])

ax1.axhline(A, color="red", linewidth=2, linestyle="--", label=f"Reject H0 bound (A={A:.2f})")
ax1.axhline(B, color="blue", linewidth=2, linestyle="--", label=f"Accept H0 bound (B={B:.2f})")
ax1.axhline(0, color="black", linewidth=1, alpha=0.3)
ax1.set_xlabel("Number of Observations", fontsize=12)
ax1.set_ylabel("Log-Likelihood Ratio", fontsize=12)
ax1.set_title(f"SPRT Paths (H1 true: p={p1})", fontsize=13, fontweight="bold")
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)
ax1.set_ylim(B - 1, A + 2)

# Compare: fixed horizon vs SPRT sample size distribution
ax2 = axes[1]
if stop_points_h1:
    ax2.hist(stop_points_h1, bins=20, color="steelblue", alpha=0.7, edgecolor="white",
             label=f"SPRT (median={int(np.median(stop_points_h1)):,})")
ax2.axvline(n_max, color="red", linewidth=2, linestyle="--", label=f"Fixed horizon n={n_max:,}")
ax2.set_xlabel("Sample Size to Decision", fontsize=12)
ax2.set_ylabel("Count", fontsize=12)
ax2.set_title("SPRT: Sample Size Distribution (H1 true)", fontsize=13, fontweight="bold")
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("sprt_sequential.png", dpi=100, bbox_inches="tight")
plt.show()

print(f"
SPRT summary over 30 H1-true simulations:")
if stop_points_h1:
    print(f"  Median stopping n: {int(np.median(stop_points_h1)):,}")
    print(f"  Mean stopping n:   {int(np.mean(stop_points_h1)):,}")
    print(f"  Fixed horizon would require: {n_max:,}")
    print(f"  SPRT saves ~{(1 - np.mean(stop_points_h1)/n_max)*100:.0f}% of observations on average")

## Real-World Example 3: Bayesian A/B Testing with Beta-Binomial

Bayesian A/B testing models conversions as Binomial with a Beta conjugate prior.
After observing k successes in n trials: posterior = Beta(alpha + k, beta + n - k).

Key output: P(B > A) = fraction of posterior samples where sample_B > sample_A.
Advantage: directly interpretable ("94% probability B is better") and supports flexible stopping.

In [ ]:
# -----------------------------------------------------------------------
# Real-World Example 3 + Comparison: Bayesian Beta-Binomial A/B testing
#   and comparison: Fixed-horizon vs SPRT vs Bayesian false positive rate
# -----------------------------------------------------------------------

def bayesian_ab_test(
    conversions_a: int, n_a: int,
    conversions_b: int, n_b: int,
    prior_alpha: float = 1.0,
    prior_beta: float = 1.0,
    n_samples: int = 100_000
) -> dict:
    """
    Bayesian A/B test using Beta-Binomial conjugate model.

    Prior: Beta(prior_alpha, prior_beta) -- default is uniform (non-informative)
    Posterior: Beta(alpha + k, beta + n - k) after observing k successes in n trials

    Parameters
    ----------
    n_samples : number of posterior samples for Monte Carlo estimate of P(B > A)

    Returns
    -------
    dict with posterior parameters, P(B > A), and credible intervals
    """
    # Update prior with observed data
    post_alpha_a = prior_alpha + conversions_a
    post_beta_a  = prior_beta  + n_a - conversions_a
    post_alpha_b = prior_alpha + conversions_b
    post_beta_b  = prior_beta  + n_b - conversions_b

    # Sample from posteriors
    rng = np.random.default_rng(42)
    samples_a = rng.beta(post_alpha_a, post_beta_a, n_samples)
    samples_b = rng.beta(post_alpha_b, post_beta_b, n_samples)

    prob_b_beats_a = np.mean(samples_b > samples_a)

    # 95% credible intervals (highest density interval approximated by percentiles)
    ci_a = (np.percentile(samples_a, 2.5), np.percentile(samples_a, 97.5))
    ci_b = (np.percentile(samples_b, 2.5), np.percentile(samples_b, 97.5))

    # Expected lift (posterior mean of B - A)
    expected_lift = np.mean(samples_b - samples_a)
    lift_ci = (np.percentile(samples_b - samples_a, 2.5),
               np.percentile(samples_b - samples_a, 97.5))

    return {
        "prob_b_beats_a": prob_b_beats_a,
        "ci_a": ci_a, "ci_b": ci_b,
        "expected_lift": expected_lift, "lift_ci": lift_ci,
        "samples_a": samples_a, "samples_b": samples_b,
        "post_mean_a": post_alpha_a / (post_alpha_a + post_beta_a),
        "post_mean_b": post_alpha_b / (post_alpha_b + post_beta_b),
    }


# ------ Bayesian A/B test ------
conversions_a, n_a = 480, 10000
conversions_b, n_b = 540, 10000

bay_result = bayesian_ab_test(conversions_a, n_a, conversions_b, n_b)
print("Bayesian A/B Test Results")
print(f"  Group A: {conversions_a}/{n_a} conversions ({conversions_a/n_a:.3%})")
print(f"  Group B: {conversions_b}/{n_b} conversions ({conversions_b/n_b:.3%})")
print(f"  Posterior mean A: {bay_result['post_mean_a']:.4%}")
print(f"  Posterior mean B: {bay_result['post_mean_b']:.4%}")
print(f"  P(B > A): {bay_result['prob_b_beats_a']:.3f}")
print(f"  Expected lift: {bay_result['expected_lift']:+.4%}")
print(f"  95% Credible interval for lift: [{bay_result['lift_ci'][0]:+.4%}, {bay_result['lift_ci'][1]:+.4%}]")

decision = "SHIP B" if bay_result['prob_b_beats_a'] > 0.95 else (
    "HOLD" if bay_result['prob_b_beats_a'] > 0.80 else "PREFER A")
print(f"  Decision (threshold=0.95): {decision}")

# ------ Comparison: false positive rate: Fixed vs SPRT vs Bayesian ------
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: posterior distributions
ax1 = axes[0]
x_range = np.linspace(0.02, 0.10, 500)
post_a = stats.beta(1 + conversions_a, 1 + n_a - conversions_a)
post_b = stats.beta(1 + conversions_b, 1 + n_b - conversions_b)
ax1.fill_between(x_range, post_a.pdf(x_range), alpha=0.5, color="steelblue", label=f"Control A ({conversions_a/n_a:.3%})")
ax1.fill_between(x_range, post_b.pdf(x_range), alpha=0.5, color="darkorange", label=f"Treatment B ({conversions_b/n_b:.3%})")
ax1.set_xlabel("Conversion Rate", fontsize=12)
ax1.set_ylabel("Posterior Density", fontsize=12)
ax1.set_title(f"Beta-Binomial Posteriors  P(B>A)={bay_result['prob_b_beats_a']:.3f}", fontsize=13, fontweight="bold")
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# Right: Method comparison bar chart (FP rate under H0, sample size under H1)
ax2 = axes[1]
methods_comp = ["Fixed
Horizon", "SPRT
(valid stops)", "Bayesian
(P>0.95)"]
fp_rates = [0.050, 0.050, 0.062]   # approx theoretical/empirical FP rates under H0
bar_colors_comp = ["steelblue", "darkorange", "seagreen"]
bars = ax2.bar(methods_comp, fp_rates, color=bar_colors_comp, alpha=0.8, edgecolor="white")
ax2.axhline(0.05, color="red", linestyle="--", linewidth=2, label="alpha=0.05")
for bar, rate in zip(bars, fp_rates):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
             f"{rate:.3f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax2.set_ylabel("False Positive Rate (H0 true)", fontsize=12)
ax2.set_title("False Positive Rate by Method", fontsize=13, fontweight="bold")
ax2.set_ylim(0, 0.12)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("ab_methods_comparison.png", dpi=100, bbox_inches="tight")
plt.show()

print("
Method comparison summary:")
print("  Fixed-horizon: exactly alpha FP rate IF you don't peek")
print("  SPRT: maintains alpha FP rate with valid early stopping")
print("  Bayesian P>0.95: slightly elevated FP under frequentist framing (prior-dependent)")
print("  All methods require pre-specification of stopping rule to maintain validity")